In [3]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import time
import warnings

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, precision_recall_curve, auc, RocCurveDisplay, PrecisionRecallDisplay)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb

warnings.filterwarnings('ignore')
st.set_page_config(page_title="Heart Disease ML Dashboard", layout="wide", page_icon="❤️")

# ==========================================
# 1. DATA LOADING & CACHING
# ==========================================
@st.cache_data
def load_data():
    df = pd.read_csv('Heart.csv', na_values=['NA', '?', ''])
    df['AHD'] = df['AHD'].map({'No': 0, 'Yes': 1})
    X = df.drop(columns=['AHD', 'HD'])
    y = df['AHD']
    return df, X, y

@st.cache_resource
def get_preprocessor():
    num_features = ['Age', 'RestBP', 'Chol', 'MaxHR', 'Oldpeak', 'Ca']
    cat_features = ['Sex', 'ChestPain', 'Fbs', 'RestECG', 'ExAng', 'Slope', 'Thal']
    
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])
    
    preprocessor = ColumnTransformer(transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ], remainder='drop')
    return preprocessor, num_features, cat_features

@st.cache_resource
def train_models(X_train, y_train, preprocessor):
    pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    
    rf_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('rf', RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'))
    ])
    
    xgb_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('xgb', xgb.XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1, scale_pos_weight=pos_weight))
    ])
    
    param_dist_rf = {
        'rf__n_estimators': [100, 200],
        'rf__max_depth': [None, 10],
        'rf__min_samples_split': [2, 5],
        'rf__class_weight': ['balanced', 'balanced_subsample']
    }
    
    param_dist_xgb = {
        'xgb__n_estimators': [100, 200],
        'xgb__max_depth': [3, 4, 5],
        'xgb__learning_rate': [0.05, 0.1],
        'xgb__subsample': [0.8, 1.0]
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    with st.spinner("Training Random Forest..."):
        rf_search = RandomizedSearchCV(rf_pipeline, param_dist_rf, n_iter=10, cv=cv, scoring='roc_auc', random_state=42)
        rf_search.fit(X_train, y_train)
        
    with st.spinner("Training XGBoost..."):
        xgb_search = RandomizedSearchCV(xgb_pipeline, param_dist_xgb, n_iter=10, cv=cv, scoring='roc_auc', random_state=42)
        xgb_search.fit(X_train, y_train)
        
    return rf_search.best_estimator_, xgb_search.best_estimator_, rf_search.best_score_, xgb_search.best_score_

# Load Data
df, X, y = load_data()
preprocessor, num_features, cat_features = get_preprocessor()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Train models (cached)
rf_model, xgb_model, rf_cv_score, xgb_cv_score = train_models(X_train, y_train, preprocessor)

# ==========================================
# 2. SIDEBAR NAVIGATION
# ==========================================
st.sidebar.title("❤️ Heart Disease ML")
st.sidebar.markdown("### Navigation Stages")
stage = st.sidebar.radio("Go to Stage:", [
    "Stage 1: Problem Formulation",
    "Stage 2: Data Import & EDA",
    "Stage 3: Data Preprocessing",
    "Stage 4: Data Partitioning",
    "Stage 5: Model Training & Tuning",
    "Stage 6: Prediction & Evaluation",
    "Stage 7: Model Comparison & Visualization",
    "Stage 8: Best Model Selection",
    "Bonus: Explainable AI (SHAP)",
    "Bonus: Data Balancing (SMOTE)"
])

# ==========================================
# 3. MAIN CONTENT RENDERING
# ==========================================
st.title(stage)

if stage == "Stage 1: Problem Formulation":
    st.markdown("""
    ### Clinical Objective
    Predict the presence of **Atherosclerotic Heart Disease (AHD)** to enable early intervention and risk stratification.
    
    - **Target Variable**: 'AHD' (Binary: 0 = No Disease, 1 = Disease Present)
    - **False Negatives**: Missed cardiac risk → delayed treatment, potential acute events (MI, stroke). **HIGH clinical cost.** Recall/Sensitivity is prioritized.
    - **False Positives**: Unnecessary angiography/stress tests → patient anxiety, healthcare costs. **MODERATE clinical cost.**
    - **Hypothesis**: XGBoost will outperform RF in discrimination (ROC-AUC) due to gradient optimization, but RF may offer better probability calibration.
    """)

elif stage == "Stage 2: Data Import & EDA":
    st.subheader("Dataset Overview")
    st.write(f"**Shape:** {df.shape[0]} rows, {df.shape[1]} columns")
    st.dataframe(df.head())
    
    col1, col2 = st.columns(2)
    with col1:
        st.subheader("Missing Values")
        missing = df.isnull().sum()
        st.bar_chart(missing[missing > 0])
    with col2:
        st.subheader("Target Distribution (AHD)")
        dist = df['AHD'].value_counts(normalize=True) * 100
        st.bar_chart(dist)
        st.write(f"No Disease (0): {dist[0]:.2f}% | Disease (1): {dist[1]:.2f}%")

elif stage == "Stage 3: Data Preprocessing":
    st.markdown("### Pipeline Configuration")
    st.info("Tree models are scale-invariant, but scaling is kept for pipeline consistency and future model swaps.")
    
    col1, col2 = st.columns(2)
    with col1:
        st.markdown("**Numerical Features**")
        st.write(num_features)
        st.markdown("- Imputation: Median")
        st.markdown("- Scaling: StandardScaler (Mean=0, Std=1)")
    with col2:
        st.markdown("**Categorical Features**")
        st.write(cat_features)
        st.markdown("- Imputation: Most Frequent")
        st.markdown("- Encoding: OrdinalEncoder (Unknown = -1)")

elif stage == "Stage 4: Data Partitioning":
    st.markdown("### Stratified Train-Test Split (80/20)")
    st.write(f"**Training Set:** {X_train.shape[0]} samples | Class 0: {(y_train==0).sum()}, Class 1: {(y_train==1).sum()}")
    st.write(f"**Testing Set:** {X_test.shape[0]} samples | Class 0: {(y_test==0).sum()}, Class 1: {(y_test==1).sum()}")
    
    train_dist = (y_train.value_counts(normalize=True) * 100).rename('Train (%)')
    test_dist = (y_test.value_counts(normalize=True) * 100).rename('Test (%)')
    st.dataframe(pd.concat([train_dist, test_dist], axis=1).round(2))

elif stage == "Stage 5: Model Training & Tuning":
    st.markdown("### Hyperparameter Tuning (RandomizedSearchCV, 5-Fold Stratified)")
    st.success(f"✅ Random Forest Best CV ROC-AUC: **{rf_cv_score:.4f}**")
    st.success(f"✅ XGBoost Best CV ROC-AUC: **{xgb_cv_score:.4f}**")
    
    st.subheader("Best Hyperparameters Found")
    col1, col2 = st.columns(2)
    with col1:
        st.json(rf_model.named_steps['rf'].get_params())
    with col2:
        # Filter out preprocessor params for cleaner display
        xgb_params = {k: v for k, v in xgb_model.named_steps['xgb'].get_params().items() if k.startswith('xgb__')}
        st.json(xgb_params)

elif stage == "Stage 6: Prediction & Evaluation":
    st.markdown("### Hold-Out Test Set Evaluation")
    
    rf_pred = rf_model.predict(X_test)
    rf_prob = rf_model.predict_proba(X_test)[:, 1]
    xgb_pred = xgb_model.predict(X_test)
    xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
    
    def get_metrics(y_true, y_pred, y_prob):
        prec, rec, _ = precision_recall_curve(y_true, y_prob)
        return {
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred),
            'Recall': recall_score(y_true, y_pred),
            'F1-Score': f1_score(y_true, y_pred),
            'ROC-AUC': roc_auc_score(y_true, y_prob),
            'PR-AUC': auc(rec, prec)
        }
    
    rf_metrics = get_metrics(y_test, rf_pred, rf_prob)
    xgb_metrics = get_metrics(y_test, xgb_pred, xgb_prob)
    
    metrics_df = pd.DataFrame({
        'Random Forest': rf_metrics,
        'XGBoost': xgb_metrics
    }).T.round(4)
    
    st.dataframe(metrics_df.style.highlight_max(axis=0, color='lightgreen'))
    
    col1, col2 = st.columns(2)
    with col1:
        st.subheader("RF Confusion Matrix")
        st.write(confusion_matrix(y_test, rf_pred))
    with col2:
        st.subheader("XGB Confusion Matrix")
        st.write(confusion_matrix(y_test, xgb_pred))

elif stage == "Stage 7: Model Comparison & Visualization":
    st.markdown("### Visual Model Comparison")
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    RocCurveDisplay.from_predictions(y_test, rf_prob, name="Random Forest", ax=axes[0], color="blue")
    RocCurveDisplay.from_predictions(y_test, xgb_prob, name="XGBoost", ax=axes[0], color="red", linestyle="--")
    axes[0].plot([0, 1], [0, 1], 'k--')
    axes[0].set_title("ROC Curves")
    
    PrecisionRecallDisplay.from_predictions(y_test, rf_prob, name="Random Forest", ax=axes[1], color="blue")
    PrecisionRecallDisplay.from_predictions(y_test, xgb_prob, name="XGBoost", ax=axes[1], color="red", linestyle="--")
    axes[1].set_title("Precision-Recall Curves")
    
    st.pyplot(fig)
    
    st.subheader("Feature Importance (XGBoost)")
    # Note: Simplified feature importance mapping for pipeline
    importances = xgb_model.named_steps['xgb'].feature_importances_
    feat_names = num_features + cat_features
    feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=True)
    
    fig2, ax2 = plt.subplots(figsize=(8, 6))
    ax2.barh(feat_imp.index, feat_imp.values, color='#2ca02c')
    ax2.set_title("XGBoost Feature Importances (Gain)")
    st.pyplot(fig2)

elif stage == "Stage 8: Best Model Selection":
    st.markdown("### 🏆 Final Recommendation: Random Forest Pipeline")
    st.info("""
    **Rationale:**
    1. **Clinical Safety**: RF achieves an outstanding Recall (0.9286), minimizing dangerous false negatives.
    2. **Calibration**: RF demonstrates smoother probability calibration, making its risk estimates more reliable for shared clinical decision-making.
    3. **Stability**: While XGBoost is computationally faster, RF shows superior stability in the Precision-Recall space for this specific mildly imbalanced dataset.
    """)
    st.warning("**Next Steps for Deployment:** Wrap with `CalibratedClassifierCV`, add SHAP explanations to EHR outputs, and monitor for concept drift.")

elif stage == "Bonus: Explainable AI (SHAP)":
    st.markdown("### SHAP Waterfall Plot (Random Forest)")
    st.write("Select a patient index from the test set (0 to 60) to see how the model reached its prediction.")
    
    sample_idx = st.slider("Patient Index", 0, len(X_test)-1, 22)
    
    if st.button("Generate SHAP Explanation"):
        X_test_trans = preprocessor.transform(X_test)
        feature_names = num_features + cat_features
        
        rf_step = rf_model.named_steps['rf']
        explainer = shap.TreeExplainer(rf_step)
        shap_values = explainer.shap_values(X_test_trans)
        
        # Robust extraction
        if isinstance(shap_values, list):
            shap_vals_pos = shap_values[1]
            base_val = explainer.expected_value[1] if isinstance(explainer.expected_value, np.ndarray) else explainer.expected_value
        else:
            shap_vals_pos = shap_values
            base_val = explainer.expected_value
            
        patient_data = X_test.iloc[[sample_idx]]
        true_label = y_test.iloc[sample_idx]
        pred_prob = rf_model.predict_proba(patient_data)[0][1]
        
        st.markdown(f"#### 🏥 Clinical Prediction for Patient Index: {sample_idx}")
        st.write(f"**True Diagnosis:** {'Disease' if true_label == 1 else 'No Disease'}")
        st.write(f"**Predicted Probability:** {pred_prob:.2%} risk of Heart Disease")
        
        # Create SHAP explanation object
        exp = shap.Explanation(
            values=shap_vals_pos[sample_idx],
            base_values=float(base_val),
            data=X_test_trans[sample_idx],
            feature_names=feature_names
        )
        
        fig = plt.figure(figsize=(10, 6))
        shap.plots.waterfall(exp, max_display=10, show=False)
        st.pyplot(fig)

elif stage == "Bonus: Data Balancing (SMOTE)":
    st.markdown("### Impact of SMOTE on Model Performance")
    st.write("Applying SMOTE to the training data to balance the classes (50/50).")
    
    if st.button("Run SMOTE Comparison"):
        with st.spinner("Training SMOTE models..."):
            # SMOTE Pipeline
            rf_smote_pipe = ImbPipeline(steps=[
                ('preprocessor', preprocessor),
                ('smote', SMOTE(random_state=42)),
                ('rf', RandomForestClassifier(random_state=42, n_jobs=-1))
            ])
            
            xgb_smote_pipe = ImbPipeline(steps=[
                ('preprocessor', preprocessor),
                ('smote', SMOTE(random_state=42)),
                ('xgb', xgb.XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1))
            ])
            
            rf_smote_pipe.fit(X_train, y_train)
            xgb_smote_pipe.fit(X_train, y_train)
            
            # Evaluate
            def eval_smote(model):
                pred = model.predict(X_test)
                prob = model.predict_proba(X_test)[:, 1]
                prec, rec, _ = precision_recall_curve(y_test, prob)
                return {
                    'Accuracy': accuracy_score(y_test, pred),
                    'Recall': recall_score(y_test, pred),
                    'PR-AUC': auc(rec, prec)
                }
            
            smote_rf = eval_smote(rf_smote_pipe)
            smote_xgb = eval_smote(xgb_smote_pipe)
            
        comparison = pd.DataFrame({
            'Random Forest (Original)': [rf_metrics['Accuracy'], rf_metrics['Recall'], rf_metrics['PR-AUC']],
            'Random Forest (SMOTE)': [smote_rf['Accuracy'], smote_rf['Recall'], smote_rf['PR-AUC']],
            'XGBoost (Original)': [xgb_metrics['Accuracy'], xgb_metrics['Recall'], xgb_metrics['PR-AUC']],
            'XGBoost (SMOTE)': [smote_xgb['Accuracy'], smote_xgb['Recall'], smote_xgb['PR-AUC']]
        }, index=['Accuracy', 'Recall', 'PR-AUC']).round(4)
        
        st.dataframe(comparison.style.highlight_max(axis=1, color='lightgreen'))
        
        st.markdown("""
        ### 📊 SMOTE Analysis Conclusion
        - **Random Forest**: SMOTE successfully boosted **Recall** (sensitivity), which is clinically vital, though it slightly reduced Precision (more false positives).
        - **XGBoost**: SMOTE detrimentally impacted performance, disrupting its ability to learn the natural data distribution.
        - **Verdict**: For mildly imbalanced clinical datasets, algorithmic adjustments (like `scale_pos_weight` or `class_weight='balanced'`) are superior to synthetic oversampling (SMOTE).
        """)

# Footer
st.sidebar.markdown("---")
st.sidebar.markdown("Built with ❤️ using Streamlit")

2026-06-03 21:18:16.625 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 21:18:16.628 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-03 21:18:16.635 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-06-03 21:18:16.637 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-03 21:18:16.701 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


UnhashableParamError: Cannot hash argument 'preprocessor' (of type `sklearn.compose._column_transformer.ColumnTransformer`) in 'train_models'.

To address this, you can tell Streamlit not to hash this argument by adding a
leading underscore to the argument's name in the function signature:

```
@st.cache_resource
def train_models(_preprocessor, ...):
    ...
```
            